## The Shared-Prime Attack (Heninger et al., 2012)

In 2012, Heninger, Durumeric, Wustrow, and Halderman scanned ~11 million TLS and
SSH public keys from the open internet. They found that **approximately 0.2% of
RSA public keys shared a prime factor** with at least one other key.

The attack is trivial: if two moduli N₁ = p·q₁ and N₂ = p·q₂ share the prime p,
then **gcd(N₁, N₂) = p** instantly yields both private keys.

The root cause was entropy-starved random number generators on embedded devices
(routers, firewalls, VPN appliances) that generated weak primes at boot time before
collecting sufficient entropy.

**Batch GCD (Bernstein, 2004)** scales this to millions of keys in O(N log² N)
time — far better than the O(N²) pairwise approach.

**Reference:** Heninger et al. (2012). *Mining Your Ps and Qs.*
USENIX Security Symposium. https://factorable.net/paper.html

In [ ]:
import json
from pathlib import Path
from Crypto.PublicKey import RSA

KEY_DIR = Path("../data/sample_keys")

manifest  = json.loads((KEY_DIR / "manifest.json").read_text())
pem_files = sorted(KEY_DIR.glob("*.pem"))

moduli    = []
filenames = []
for pem_path in pem_files:
    key = RSA.import_key(pem_path.read_bytes())
    moduli.append(int(key.n))
    filenames.append(pem_path.name)

print(f"Loaded {len(moduli)} public keys")
print(f"Ground-truth vulnerable keys in manifest: {len(manifest)}")
print()
for fname in sorted(manifest)[:3]:
    print(f"  {fname}: n={manifest[fname]['n'][:20]}...")

In [ ]:
import time
from rsa_attacks.common_factor import batch_gcd

start   = time.perf_counter()
gcds    = batch_gcd(moduli)
elapsed = time.perf_counter() - start

vuln_idx = [i for i, g in enumerate(gcds) if g > 1]

print(f"batch_gcd on {len(moduli)} x 1024-bit keys: {elapsed:.3f}s")
print(f"\nVulnerable indices found : {vuln_idx}")
print(f"Expected from manifest   : {sorted(int(k[4:7]) for k in manifest)}")
print(f"\nAll correct: {set(vuln_idx) == {int(k[4:7]) for k in manifest}}")

In [ ]:
from rsa_attacks.common_factor import attack

results    = attack(moduli)
r          = results[0]                   # first recovered key
fname      = filenames[r["index"]]
truth      = manifest[fname]
expected_p = int(truth["p"], 16)

print(f"Key file : {fname}")
print(f"Index    : {r['index']}")
print()
print(f"Recovered p : {hex(r['p'])[:26]}...")
print(f"Manifest  p : {truth['p'][:26]}...")
print()
print(f"p x q == n  : {r['p'] * r['q'] == r['n']}")
print(f"p matches   : {r['p'] == expected_p or r['q'] == expected_p}")
print()
print(f"{'File':<16}  {'p (truncated)':<30}  q (truncated)")
print("-" * 82)
for res in results:
    fn = filenames[res["index"]]
    print(f"{fn:<16}  {hex(res['p'])[:28]:<30}  {hex(res['q'])[:28]}")

## Real-World Impact

The 10 vulnerable keys above share a single prime factor `p`. In a real deployment,
recovering `p` immediately yields the private key:

```
phi(n) = (p - 1)(q - 1)
d      = e^{-1} mod phi(n)   ← private exponent
```

An attacker with access to the public keys could:
1. Decrypt all recorded TLS sessions using the compromised keys.
2. Forge RSA signatures on arbitrary documents.

Heninger et al. reported **64 000 TLS hosts** and **108 000 SSH hosts** were
immediately vulnerable. Most were network devices (printers, routers, firewalls)
that generated RSA keys at factory reset without sufficient entropy.

**Mitigations:**
- Use a hardware RNG or seed the PRNG from multiple entropy sources before keygen.
- On Linux, `/dev/urandom` is safe after boot; many embedded systems now use
  factory-provisioned seeds or a TPM for initial entropy.
- Audit existing deployed keys for shared factors periodically.